# OSRT-605M midtrain3 — Colab GPU (drive from VSCode)

Continued pretraining of the v6 base toward Chinchilla-optimal. Resumes the
**same 12,600-step cosine** from wherever it left off — the latest checkpoint on
the private HF repo `HallD/osrt-v6-ckpt` (currently `step_2300`).

**How to run:** connect VSCode to a Colab runtime (Colab extension → *Connect to
Colab* → pick an **H100** or **A100** GPU runtime), then run these cells top to
bottom. The kernel lives on the Colab VM, so training runs on its GPU.

### Cross-session persistence (the important part)
The Colab VM disk is **ephemeral** and sessions have a 24h cap. `--hf-repo` makes
the run:
1. **pull** the latest `midtrain3_step_*` (+ the base) from HF on start, and
2. **push** every new checkpoint to HF as it saves (every 100 steps).

So a disconnect / reclaim loses at most ~100 steps — the next session's *Full run*
cell pulls the latest and continues the identical cosine. **Nothing lives only on
the VM.**

### Cost reality (read once)
This is a multi-billion-token pretraining grind: ~10,000 steps still to go to
reach 1× Chinchilla (~step 12,600). At Colab GPU rates that is **many sessions**.
Bank checkpoints, watch W&B, and **always run the Stop cell** when you step away —
idle GPU runtimes burn units. If a run misbehaves, stop it; the HF checkpoints are
safe regardless.

## 1 · GPU check
Confirm a CUDA GPU is attached before spending anything. If this shows *no GPU*,
reconnect the runtime to a GPU type and re-run.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv || echo 'NO GPU — attach a GPU runtime'
import torch, sys
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(cpu)')
assert torch.cuda.is_available(), 'Attach a GPU runtime before continuing.'

## 2 · Clone the repo + install pinned deps
The tokenizer (`v6_tokenizer_export/`) is committed, so a clone is all that's
needed. Versions are pinned to match the training code (torch 2.11 lineage).

In [ ]:
%cd /content
![ -d osrt ] || git clone https://github.com/CodeHalwell/OSRT-605M-A269M.git osrt
%cd /content/osrt
!git pull --ff-only 2>/dev/null; true
!pip install -q \
  transformers==5.3.0 datasets==4.6.1 tokenizers==0.22.2 safetensors==0.7.0 \
  wandb==0.25.1 lion-pytorch==0.2.4 huggingface_hub
print('repo + deps ready')

## 3 · Secrets
Paste your tokens. **Do not commit this notebook with tokens filled in** — clear
them before saving. `HF_TOKEN` needs write access to `HallD/osrt-v6-ckpt`.

> Tip: in Colab you can instead use the 🔑 *Secrets* panel and read them with
> `from google.colab import userdata; userdata.get('HF_TOKEN')` — safer than
> pasting into a cell.

In [ ]:
import os
# Prefer Colab's Secrets panel (🔑 in the left sidebar) — add HF_TOKEN and
# WANDB_API_KEY there ONCE; they never touch this file or git.
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    print('loaded tokens from Colab Secrets')
except Exception:
    # Fallback ONLY if not on Colab: paste TEMPORARILY, then clear before saving.
    os.environ.setdefault('HF_TOKEN', '')       # <-- leave empty; use Secrets panel
    os.environ.setdefault('WANDB_API_KEY', '')  # <-- leave empty; use Secrets panel
os.environ['PYTHONPATH'] = 'src'
# Guards learned the hard way:
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '30'   # don't hang forever on a slow HF pull
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
assert os.environ.get('HF_TOKEN') and os.environ.get('WANDB_API_KEY'), \
    'Set HF_TOKEN + WANDB_API_KEY in the Colab Secrets panel (🔑).'
print('secrets set')

## 4 · Sanity gate — 30 steps
Always run this **first** on a fresh session. It exercises the full stack
(model build, data streams, optimizer, one checkpoint) in ~2-3 min. If it errors,
fix that before the full run — don't burn GPU discovering a broken stream at step 500.

`--hf-repo` pulls the checkpoints from HF on first run (the base + latest `step_*`,
~10GB) — you pay this download **once**; the full run below reuses the cached files.
Sanity uses its own `midtrain3-sanity` checkpoint prefix, so it runs a **clean
30-step pass from the base** and does not touch the real run's progress or push to HF.

`--num-workers 0` is **mandatory on Colab**: the spawned streaming workers hit a
fatal `PyGILState_Release` teardown race that kills the run mid-stream.

In [ ]:
!python scripts/lightning_midtrain3.py --sanity \
  --ckpt-dir /content/ckpt \
  --tokenizer v6_tokenizer_export \
  --hf-repo HallD/osrt-v6-ckpt \
  --num-workers 0

## 5 · Full run — resumes from the latest HF checkpoint
This runs the real cosine. `--hf-repo` pulls the latest `midtrain3_step_*` (+ base)
from HF, the resume-scan continues the same 12,600-step schedule, and each new
checkpoint is pushed back to HF every 100 steps.

The cell **streams live output** and blocks while training — that's expected;
watch the `step N/12600 | task … | tok/s …` lines. If the VSCode connection drops,
the Colab kernel keeps running server-side; reconnect and the output reattaches.
If the *VM* is reclaimed, just re-run this cell next session — it resumes from HF.

Config baked in (`MidtrainExtend3Config`): seq 4096, eff-batch 66, peak LR 5e-5,
0.75 reasoning-dense mix, in-loop eval **off** (its dataset build stalls the GPU
and triggers idle reclaims), checkpoint every 100.

In [ ]:
!python scripts/lightning_midtrain3.py \
  --ckpt-dir /content/ckpt \
  --tokenizer v6_tokenizer_export \
  --hf-repo HallD/osrt-v6-ckpt \
  --ckpt-interval 100 \
  --num-workers 0

### Optional: run detached so it survives a VSCode disconnect cleanly
If you'd rather not hold the cell open, launch it in the background on the VM and
tail the log. Re-running the tail cell re-attaches to the live output.

In [ ]:
# launch detached (run once)
import subprocess, os
env = dict(os.environ)
subprocess.Popen(
    'python scripts/lightning_midtrain3.py --ckpt-dir /content/ckpt '
    '--tokenizer v6_tokenizer_export --hf-repo HallD/osrt-v6-ckpt '
    '--ckpt-interval 100 --num-workers 0 > /content/mt3.log 2>&1 &',
    shell=True, env=env)
print('launched in background → /content/mt3.log')

In [ ]:
# tail the background log (re-run anytime to see progress)
!tail -n 40 /content/mt3.log

## 6 · Monitor
- **W&B**: the `osrt-v6-midtrain3` run — watch `extend/task_loss` and `tok/s`.
- **HF**: `HallD/osrt-v6-ckpt` gains a new `osrt_v5_midtrain3_step_*.pt` every 100
  steps (the daemon prunes to the newest 3 remotely).
- Healthy signs: `drop=0`, `task` loss drifting down, `tok/s` steady (~7-9k on H100).

In [ ]:
# quick HF check — what's the latest banked checkpoint?
from huggingface_hub import HfApi
import re, os
fs = [f for f in HfApi().list_repo_files('HallD/osrt-v6-ckpt', repo_type='model',
                                          token=os.environ['HF_TOKEN'])
      if 'midtrain3_step' in f]
fs.sort(key=lambda f: int(re.search(r'(\d+)', f).group(1)))
print('latest on HF:', fs[-1] if fs else '(none)')

## 6b · (Optional) Held-out in-distribution perplexity
Measures perplexity on a **held-out slice of Nemotron-CC-Math** — the dominant
reasoning source midtrain3 actually trains on — read from a `skip=2M` offset so
the samples are genuinely unseen (well past the full run's ~0.57M-record
consumption from this set).

**Why this and not FineWeb:** FineWeb is only ~15% of the current mix, and
training on 75% reasoning data makes its ppl *drift up* (distribution shift),
which is muddy to read. This eval is **in-distribution** — it measures the thing
we're pushing — so the number is the honest "is pretraining helping" signal.

**When to run:** occasionally (e.g. once a session, around step ~5,000+), after
**stopping training** (VRAM contention). First call pays a short skip cost
(<~1 min — the set is huge, skip is cheap).

**Interpretation:** there's **no prior baseline** (new metric) — the first run
sets it, so record the number. Across sessions it should **trend DOWN** as
pretraining improves the model on this distribution. Flat or rising = diminishing
returns (or a problem). Still **not** a capability test — GSM8K waits for the SFT
re-run; this is the base-model signal.

In [ ]:
# Held-out IN-DISTRIBUTION perplexity: Nemotron-CC-Math (the dominant reasoning
# slice midtrain3 actually trains on), read from a skip offset PAST the training
# budget so the samples are genuinely unseen. This is the honest "is pretraining
# helping" signal — unlike FineWeb (a 15% minority slice), ppl here should DROP
# as training progresses. STOP training first (VRAM contention).
import os, glob, re, math, torch
from transformers import AutoTokenizer
from osrt.presets import build_config
from osrt.model import OSRTForCausalLM
from osrt.data import make_loader

CKPT_DIR = '/content/ckpt'
cks = glob.glob(f'{CKPT_DIR}/osrt_v5_midtrain3_step_*.pt') + \
      glob.glob(f'{CKPT_DIR}/osrt_v5_midtrain3_rescue_step_*.pt')
assert cks, f'no midtrain3 checkpoint in {CKPT_DIR} - run/resume the training cell first'
ckpt_path = max(cks, key=lambda f: int(re.search(r'_step_(\d+)', f).group(1)))
step = int(re.search(r'_step_(\d+)', ckpt_path).group(1))
print(f'evaluating: {os.path.basename(ckpt_path)} (step {step})')

tok = AutoTokenizer.from_pretrained('v6_tokenizer_export')
cfg = build_config(vocab_size=len(tok), real_vocab_size=len(tok),
                   bos_token_id=tok.bos_token_id, eos_token_id=tok.eos_token_id,
                   pad_token_id=tok.pad_token_id, fused_cross_entropy_chunks=8)
device = torch.device('cuda')
model = OSRTForCausalLM(cfg).to(device)   # eager (no torch.compile needed for eval)
sd = torch.load(ckpt_path, map_location=device, weights_only=True)['model_state_dict']
missing, unexpected = model.load_state_dict(sd, strict=False)
assert not missing and not unexpected, f'state mismatch: missing={missing[:3]} unexpected={unexpected[:3]}'
model.eval()

# Held-out math/reasoning slice. skip=2,000,000 records is ~3.5x past the FULL
# 12,600-step training consumption from this set (~0.57M records) -> unseen.
# Nemotron-CC-Math is large, so the skip won't exhaust it and costs <1 min.
EVAL_STEPS, BATCH = 40, 6
loader = make_loader(
    dataset_configs=[{
        'name': 'nemotron-cc-math-heldout',
        'hf_id': 'nvidia/Nemotron-CC-Math-v1',
        'hf_config': '4plus',
        'weight': 1.0,
        'skip': 2_000_000,
    }],
    seq_len=4096, tokenizer_name='v6_tokenizer_export',
    batch_size=BATCH, step_num=999999, num_workers=0,
)
it = iter(loader)
total_loss = total_tok = 0
with torch.inference_mode():
    for _ in range(EVAL_STEPS):
        input_ids, labels = next(it)
        input_ids, labels = input_ids.to(device), labels.to(device)
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            out = model(input_ids, labels=labels)
        n = int((labels != -100).sum())
        total_loss += out.loss.item() * n; total_tok += n
mean_loss = total_loss / max(total_tok, 1)
ppl = math.exp(min(mean_loss, 20.0))
print(f"\n== held-out Nemotron-CC-Math perplexity (in-distribution) ==")
print(f"step {step} | loss {mean_loss:.4f} | ppl {ppl:.2f} | tokens {total_tok:,}")
print('NO prior baseline (new metric) -> the FIRST run sets it; record the number.')
print('READ: this is the distribution midtrain3 trains on, so ppl should TREND DOWN')
print('across sessions as pretraining helps. Flat/rising = diminishing returns or a problem.')

## 7 · STOP — always run when you step away
Idle GPU runtimes keep billing. Disconnect/delete the runtime from the VSCode
Colab panel (or Colab → *Runtime → Disconnect and delete runtime*). Checkpoints
are safe on HF, so stopping never loses progress.

In [ ]:
# kill any background training process on the VM (does not delete the runtime)
!pkill -f lightning_midtrain3 && echo 'training stopped' || echo 'nothing running'